In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from scipy.stats import norm

# =========================================================
# WEEK 9 — FUNCTION 7 (STRICT [0,1] BOUNDS)
# - Ensemble + KFold => predictive mean + uncertainty
# - Mix EI (predictable) + UCB (exploration hedge)
# - Candidate generation: global + local (both clipped to [0,1])
# - Prints x_next with 6 decimals
# =========================================================

# -----------------------------
# 1) Data: 6D inputs and 1D outputs (Function 7)
# -----------------------------
X_raw = np.array([
    [0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269, 0.79586362],
    [0.54300258, 0.9246939 , 0.34156746, 0.64648585, 0.71844033, 0.34313266],
    [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285, 0.6402654 ],
    [0.11886697, 0.61505494, 0.90581639, 0.8553003 , 0.41363143, 0.58523563],
    [0.63021764, 0.8380969 , 0.68001305, 0.73189509, 0.52673671, 0.34842921],
    [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277, 0.09579366],
    [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833, 0.73096984],
    [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418, 0.10703171],
    [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083, 0.6924164 ],
    [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917, 0.05109986],
    [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098, 0.63747637],
    [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415, 0.35995166],
    [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592, 0.82233079],
    [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175, 0.71195755],
    [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938, 0.51691776],
    [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482, 0.44967361],
    [0.41762629, 0.06409998, 0.24566877, 0.5590408 , 0.19153138, 0.25464092],
    [0.72628566, 0.46489581, 0.92457051, 0.8072454 , 0.6354384 , 0.14341787],
    [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469, 0.6190825 ],
    [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373, 0.11486924],
    [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282, 0.63370429],
    [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153, 0.95101392],
    [0.68685257, 0.04101721, 0.00757301, 0.285009  , 0.69156848, 0.6555429 ],
    [0.17597754, 0.6244165 , 0.29554198, 0.46955276, 0.09776977, 0.72814108],
    [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501, 0.73066019],
    [0.06661051, 0.52804507, 0.8160952 , 0.96101714, 0.08650933, 0.77778822],
    [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781, 0.51985176],
    [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.5523983 , 0.08130609],
    [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.7193764 , 0.36288398],
    [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.9986547 , 0.07966402],
    [1.04245 , 1.024693, 1.02457 , 1.061017, 1.098654, 1.051013],
    [0.019976, 0.432955, 0.301662, 0.169496, 0.348651, 0.743371],
    [0.611853, 0.139495, 0.292145, 0.366362, 0.45607 , 0.785175],
    [0.015006, 0.390905, 0.178469, 0.119929, 0.088415, 0.904408],
    [0.046821, 0.309546, 0.608802, 0.064364, 0.39334 , 0.990644],
    [0.011478, 0.62027 , 0.525606, 0.053535, 0.52488 , 0.666127],
    [0.028679, 0.235471, 0.148723, 0.076614, 0.11285 , 0.837107],
    [0.069198, 0.39455 , 0.352452, 0.093928, 0.370707, 0.725655]
])

y_raw = np.array([
    6.04432696e-01, 5.62753067e-01, 7.50323668e-03, 6.14243025e-02,
    2.73046801e-01, 8.37465723e-02, 1.36496830e+00, 9.26449549e-02,
    1.78695987e-02, 3.35649360e-02, 7.35163042e-02, 2.06309698e-01,
    8.82563400e-03, 2.68400317e-01, 6.11525528e-01, 1.47981826e-02,
    2.74892508e-01, 6.67632469e-02, 4.21183545e-02, 2.70146502e-03,
    1.82090730e-02, 7.01602756e-03, 1.00506611e-01, 4.75395516e-01,
    6.75141631e-01, 5.16457219e-01, 3.77747962e-03, 3.13433331e-03,
    2.13425228e-02, 9.54111589e-02, 4.636858051500375e-06, 1.680828424430851,
    1.1170576710554418, 0.44099891630237703, 0.8950628737420184,
    0.664856997347448, 0.6583383225997628, 1.6173276124769211
])

# -----------------------------
# 2) Config (Week 9)
# -----------------------------
RANDOM_STATE = 123
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

INPUT_DIM = X_raw.shape[1]

# modest compute, robust uncertainty
N_FOLDS = 5
N_ENSEMBLE = 4
EPOCHS = 800
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10
PATIENCE = 60

# candidate search (strict [0,1])
N_GLOBAL = 18000
N_LOCAL = 4000
LOCAL_SCALE = 0.08

# acquisition knobs
EI_XI = 0.01
UCB_BETA = 1.6
MIX_ALPHA = 0.65  # EI weight (predictable) vs UCB (emergent hedge)

# -----------------------------
# 3) Model
# -----------------------------
class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x)

# -----------------------------
# 4) Acquisition
# -----------------------------
def gaussian_ei(mu, sigma, f_best, xi=0.0):
    sigma = np.maximum(sigma, 1e-9)
    z = (mu - f_best - xi) / sigma
    return (mu - f_best - xi) * norm.cdf(z) + sigma * norm.pdf(z)

def gaussian_ucb(mu, sigma, beta=1.0):
    return mu + beta * np.maximum(sigma, 1e-9)

# -----------------------------
# 5) Training utilities
# -----------------------------
def train_one_model(Xtr, ytr, Xva, yva, seed):
    torch.manual_seed(seed)
    model = MLPRegressorTorch(INPUT_DIM)
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr.reshape(-1, 1), dtype=torch.float32)
    Xva_t = torch.tensor(Xva, dtype=torch.float32)
    yva_t = torch.tensor(yva.reshape(-1, 1), dtype=torch.float32)

    best_val = np.inf
    best_state = None
    bad = 0

    model.train()
    n = Xtr_t.shape[0]

    for _ in range(EPOCHS):
        idx = torch.randperm(n)
        for start in range(0, n, BATCH_SIZE):
            batch = idx[start:start + BATCH_SIZE]
            pred = model(Xtr_t[batch])
            loss = loss_fn(pred, ytr_t[batch])
            opt.zero_grad()
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            vloss = loss_fn(model(Xva_t), yva_t).item()
        model.train()

        if vloss + 1e-6 < best_val:
            best_val = vloss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    return model

def ensemble_predict(models, Xcand_scaled):
    Xc_t = torch.tensor(Xcand_scaled, dtype=torch.float32)
    preds = []
    with torch.no_grad():
        for m in models:
            preds.append(m(Xc_t).numpy().ravel())
    P = np.vstack(preds)  # [M, N]
    mu = P.mean(axis=0)
    sigma = P.std(axis=0, ddof=1) if P.shape[0] > 1 else np.full(P.shape[1], 1e-6)
    return mu, sigma

# -----------------------------
# 6) STRICT [0,1] bounds + candidates
# -----------------------------
def make_bounds_strict_unit_cube(input_dim):
    lo = np.zeros(input_dim)
    hi = np.ones(input_dim)
    return lo, hi

def generate_candidates(rng, lo, hi, x_best, n_global, n_local):
    # Global uniform in [0,1]^d
    Xg = rng.uniform(lo, hi, size=(n_global, INPUT_DIM))

    # Local Gaussian around current best, clipped to [0,1]
    Xl = x_best + rng.normal(0.0, LOCAL_SCALE, size=(n_local, INPUT_DIM))
    Xl = np.clip(Xl, lo, hi)

    return np.vstack([Xg, Xl])

# -----------------------------
# 7) Main
# -----------------------------
def main():
    # Clip training inputs too (since your raw data contains >1 values)
    X_train = np.clip(X_raw, 0.0, 1.0)

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_scaled = x_scaler.fit_transform(X_train)
    y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

    # current best observed (exploit), then clip to [0,1]
    best_idx_obs = int(np.argmax(y_raw))
    x_best = np.clip(X_train[best_idx_obs].copy(), 0.0, 1.0)

    # Train KFold x Ensemble models (robust uncertainty)
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    models = []
    base_seed = RANDOM_STATE * 1000

    split_id = 0
    for tr_idx, va_idx in kf.split(X_scaled):
        Xtr, ytr = X_scaled[tr_idx], y_scaled[tr_idx]
        Xva, yva = X_scaled[va_idx], y_scaled[va_idx]

        for e in range(N_ENSEMBLE):
            seed = base_seed + 97 * split_id + 13 * e
            models.append(train_one_model(Xtr, ytr, Xva, yva, seed))

        split_id += 1

    # strict bounds
    lo, hi = make_bounds_strict_unit_cube(INPUT_DIM)
    rng = np.random.RandomState(RANDOM_STATE)

    X_cand = generate_candidates(rng, lo, hi, x_best, N_GLOBAL, N_LOCAL)
    X_cand_scaled = x_scaler.transform(X_cand)

    # Predict distribution in scaled-y space
    mu_s, sigma_s = ensemble_predict(models, X_cand_scaled)

    # Convert to original y scale
    mu = y_scaler.inverse_transform(mu_s.reshape(-1, 1)).ravel()
    y_scale = float(y_scaler.scale_[0])
    sigma = np.maximum(sigma_s * y_scale, 1e-9)

    f_best = float(np.max(y_raw))

    ei = gaussian_ei(mu, sigma, f_best, xi=EI_XI)
    ucb = gaussian_ucb(mu, sigma, beta=UCB_BETA)

    # Stable mixing (normalize each term)
    def zscore(a):
        s = np.std(a)
        if s < 1e-12:
            return a - np.mean(a)
        return (a - np.mean(a)) / s

    score = MIX_ALPHA * zscore(ei) + (1.0 - MIX_ALPHA) * zscore(ucb)

    # pick best score; tie-break by higher mu
    top = np.where(score >= np.max(score) - 1e-12)[0]
    best_idx = int(top[np.argmax(mu[top])])

    x_next = np.clip(X_cand[best_idx], lo, hi)
    x_next = np.round(x_next, 6)

    # Print in fixed 6 decimals
    np.set_printoptions(suppress=True, formatter={"float_kind": lambda v: f"{v:.6f}"})
    print("RECOMMENDED NEXT POINT")
    print("x_next =", x_next)

if __name__ == "__main__":
    main()


RECOMMENDED NEXT POINT
x_next = [0.000000 0.367783 0.346341 0.052998 0.363011 0.730958]
